# Spikes Analysis

Analysis of spiking activity in the cerebellar SNN model during co-simulations, including:
- **Firing rate computation** for each cerebellar population (GrC, MLI, PC, CNe), pooled across regions for a given coupling strength (G) and condition
- **Comparison with experimental data** from literature (Chen et al. 2016, 2017; van der Heijden et al. 2022), plotted as simulation mean vs. experimental mean with min–max or mean ± std ranges
- **Raster plots and PSTH histograms** (20 ms bins) for individual spike trains from a single simulation, showing temporal structure of spiking for each cell type
- **Afferent coupling signal visualization**, displaying the summed cortical + subcortical input to selected cerebellar regions over time, with dominant oscillation frequency extracted via Welch's PSD

**Authors:** Alice Geminiani ([alice.geminiani@unipv.it](mailto:alice.geminiani@unipv.it)) and GitHub Copilot with Claude Opus 4.6

In [ ]:
# Load and summarize spikes pickle files from Final10reps/COSIM
import os
from pprint import pprint
import dill  # dill is used in file_utils.py for pickle files
from utils.file_utils import load_pickled_dict
from utils.plot_utils import save_figure_multi_format
from xarray import DataArray
import numpy as np 
from matplotlib import cm

data_dir = os.path.join('publication_data', 'Final10reps', 'COSIM')
abs_data_dir = os.path.abspath(data_dir)
print(f"Loading spikes pickle files from: {abs_data_dir}")

# List .pkl files in the directory
file_rates_all = "RatesFromSpikes.pkl"
file_spikes_single = "iG06_nsd4_COSIM_Spikes.pkl"


file_path_rates = os.path.join(abs_data_dir, file_rates_all)
rates_spikes_all = DataArray.from_dict(load_pickled_dict(file_path_rates))

file_path_spikes = os.path.join(abs_data_dir, file_spikes_single)
spikes_single = load_pickled_dict(file_path_spikes)


# Display a summary of each loaded pickle file
def summarize(obj):
    if isinstance(obj, dict):
        return f"dict with keys: {list(obj.keys())}"
    elif hasattr(obj, 'shape'):
        return f"type: {type(obj)}, shape: {obj.shape}"
    else:
        return f"type: {type(obj)}, str: {str(obj)[:100]}"

print(f"\nSummary for {file_rates_all}:")
print(summarize(rates_spikes_all))
print(f"\nSummary for {file_spikes_single}:")
print(summarize(spikes_single))

In [ ]:
# Reference experimental data for comparison
exp_firing_cells = {
    'mossy_fiber': [10.0, 0, 20.0],              # [mean, sem, N, min, max] Hz
    'granule_cell': [4.2, 1.1, 45, 0.0, 30.0],     # From Fig. 4d of Chen, S., Augustine, G.J. & Chadderton, P. Serial processing of kinematic signals by cerebellar circuitry during voluntary whisking. Nat Commun 8, 232 (2017). https://doi.org/10.1038/s41467-017-00312-1
    'purkinje_cell': [70.1, 4.4, 49, 23.8, 178.7],       # From Fig. 1F of Susu ChenGeorge J AugustinePaul Chadderton (2016) The cerebellum linearly encodes whisker position during voluntary movement eLife 5:e10509.             
    'MLI': [40.8, 5.9, 34, 9.3, 175.3],     # From Fig. 6c of Chen, S., Augustine, G.J. & Chadderton, P. Serial processing of kinematic signals by cerebellar circuitry during voluntary whisking. Nat Commun 8, 232 (2017). https://doi.org/10.1038/s41467-017-00312-1
    'dcn_cell_glut_large': [67.2, 4.2, 33, 43.1, 91.3]       # From Table 1 of van der Heijden, Meike E. et al. iScience, Volume 25, Issue 11, 105429
}                                                         # For DCN min and max computed as mean - std dev and min + std dev

cell_colors = {
    'mossy_fiber': 'darkblue',
    'granule_cell': 'darkred',
    'purkinje_cell': 'darkgreen',
    'MLI': 'orange',
    'dcn_cell_glut_large': 'black'
}

cell_names = {
    'mossy_fiber': 'mossy',
    'granule_cell': 'GrC',
    'purkinje_cell': 'PC',
    'MLI': 'MLI',
    'dcn_cell_glut_large': 'CNe'
}

viridis = cm.get_cmap('viridis')
color_exp = viridis(0.35)  

In [ ]:
# For G=6 and condition 'CEREBON', pool all rate values for each Population across all Regions, then compute mean and std
# Uses rates_spikes_all loaded in the first cell
import numpy as np

G_value = 6
condition_value = "CEREBON"

# Find indices for G=6 and Condition='CEREBON'
G_idx = list(rates_spikes_all.coords["G"].values).index(G_value)
cond_idx = list(rates_spikes_all.coords["Condition"].values).index(condition_value)

print(f"Statistics for G={G_value}, Condition='{condition_value}':\n")

sim_means = {}  # Save means for each population for later plotting
sim_stds = {}   # Save stds for error bars
pooled_MLI = []
for iP, pop in enumerate(rates_spikes_all.coords["Population"].values):
    data = rates_spikes_all[G_idx, cond_idx, :, iP, :].values
    pooled = data.flatten()
    pooled = pooled[~np.isnan(pooled)]  # Remove NaNs
    if pop.item() in ['basket_cell', 'stellate_cell']:
        pooled_MLI.append(pooled)
    else:
        mean_val = np.mean(pooled)
        std_val = np.std(pooled)
        sim_means[pop.item()] = mean_val  # Save for later use
        sim_stds[pop.item()] = std_val   # Save std for error bars
        print(f"{pop.item()}: Mean = {mean_val:.3f} Hz, Std = {std_val:.3f} Hz, N = {len(pooled)}")
# Merge basket and stellate into MLI
if pooled_MLI:
    pooled_MLI = np.concatenate(pooled_MLI)
    mean_val = np.mean(pooled_MLI)
    std_val = np.std(pooled_MLI)
    sim_means['MLI'] = mean_val
    sim_stds['MLI'] = std_val
    print(f"MLI (basket+stellate): Mean = {mean_val:.3f} Hz, Std = {std_val:.3f} Hz, N = {len(pooled_MLI)}")

In [ ]:
# Plot simulation vs experimental firing rates for each cell type and save figure
import matplotlib.pyplot as plt
import numpy as np
import os

# Set matplotlib rcParams for consistent font sizes
plt.rcParams.update({
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
})

# No need to merge basket and stellate, use MLI directly
exp_firing_cells_merged = exp_firing_cells.copy()

# Define desired order: granule_cell, MLI, purkinje_cell, dcn_cell_glut_large (mossy_fiber removed)
ordered_cell_types = []
for ct in ['granule_cell', 'MLI', 'purkinje_cell', 'dcn_cell_glut_large']:
    if ct in exp_firing_cells_merged:
        ordered_cell_types.append(ct)
cell_types = ordered_cell_types

sim_values = [sim_means.get(cell, np.nan) for cell in cell_types]
exp_means = [exp_firing_cells_merged[cell][0] for cell in cell_types]
exp_mins = [exp_firing_cells_merged[cell][3] if len(exp_firing_cells_merged[cell]) > 3 else exp_firing_cells_merged[cell][0] for cell in cell_types]
exp_maxs = [exp_firing_cells_merged[cell][4] if len(exp_firing_cells_merged[cell]) > 4 else exp_firing_cells_merged[cell][0] for cell in cell_types]

cell_colors_merged = cell_colors.copy()

# Use cell_names for x-axis labels
xtick_labels = [cell_names[cell] for cell in cell_types]

# Asymmetric error bars for exp: [mean - min, max - mean]
exp_yerr_lower = [m - lo for m, lo in zip(exp_means, exp_mins)]
exp_yerr_upper = [hi - m for m, hi in zip(exp_means, exp_maxs)]

color_cosim = viridis(0.75)  # Green

fig, ax = plt.subplots(figsize=(5, 5))
x = np.arange(len(cell_types))
hw = 0.25  # half-width of the horizontal line

# Simulation: horizontal line at mean value
for j, cell in enumerate(cell_types):
    ax.hlines(sim_values[j], x[j] - hw, x[j] + hw, colors=color_cosim,
              linewidth=2, zorder=3, label='Simulation' if j == 0 else None)

# Experimental: mean dot (smaller) with min-max error bars
ax.errorbar(x, exp_means, yerr=[exp_yerr_lower, exp_yerr_upper],
            fmt='o', color=color_exp, capsize=6, markersize=5, linewidth=1.5,
            label='Exp mean (min–max)')

ax.set_ylabel('Firing rate (Hz)')
ax.set_xticks(x)
ax.set_xticklabels(xtick_labels)
ax.legend(loc='upper left')
plt.tight_layout()

# Save figure to spikes_figures folder
fig_folder = os.path.join('Final10reps', 'coupling_and_spikes_figures')
os.makedirs(fig_folder, exist_ok=True)
fig_path = os.path.join(fig_folder, 'sim_vs_exp_firing_minmax.png')
plt.savefig(fig_path, dpi=300)
save_figure_multi_format(fig, os.path.join(fig_folder, 'sim_vs_exp_firing_minmax'))

plt.show()

In [ ]:
# Second plot: exp range is mean ± std (std computed from SEM and N), mossy fibers excluded
plt.rcParams.update({
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
})

exp_stds = []
for cell in cell_types:
    sem = exp_firing_cells[cell][1]
    N = exp_firing_cells[cell][2]
    std = sem * np.sqrt(N)
    exp_stds.append(std)

color_cosim = viridis(0.75)  # Green, same as Co-sim line in PSD plot

fig, ax = plt.subplots(figsize=(5, 5))
x = np.arange(len(cell_types))
hw = 0.25

# Simulation: horizontal line at mean value
for j, cell in enumerate(cell_types):
    ax.hlines(sim_values[j], x[j] - hw, x[j] + hw, colors=color_cosim,
              linewidth=2, zorder=3, label='Simulation' if j == 0 else None)

# Experimental: mean dot (smaller) with ± std error bars
ax.errorbar(x, exp_means, yerr=exp_stds,
            fmt='o', color=color_exp, capsize=6, markersize=5, linewidth=1.5,
            label='Exp mean (mean ± std)')

ax.set_ylabel('Firing rate (Hz)')
ax.set_xticks(x)
ax.set_xticklabels(xtick_labels)
ax.legend(loc='upper left')
plt.tight_layout()

# Save figure to spikes_figures folder
fig_folder = os.path.join('Final10reps', 'coupling_and_spikes_figures')
os.makedirs(fig_folder, exist_ok=True)
fig_path = os.path.join(fig_folder, 'sim_vs_exp_firing_meanstd.png')
plt.savefig(fig_path, dpi=300)
save_figure_multi_format(fig, os.path.join(fig_folder, 'sim_vs_exp_firing_meanstd'))

plt.show()

In [ ]:
# Raster, histogram (20ms bins, cell-specific y), and point plot (sim mean line + exp mean with min-max) for each main cell type
import matplotlib.pyplot as plt
import numpy as np
import os

main_cells = ['granule_cell', 'MLI', 'purkinje_cell', 'dcn_cell_glut_large']
num_cells = len(main_cells)

# Get simulation and experimental values for each cell type
sim_values = [sim_means.get(cell, np.nan) for cell in main_cells]
exp_means = [exp_firing_cells[cell][0] for cell in main_cells]
exp_mins = [exp_firing_cells[cell][3] if len(exp_firing_cells[cell]) > 3 else exp_firing_cells[cell][0] for cell in main_cells]
exp_maxs = [exp_firing_cells[cell][4] if len(exp_firing_cells[cell]) > 4 else exp_firing_cells[cell][0] for cell in main_cells]
cell_colors_merged = cell_colors.copy()

# Histogram y-limits per cell type
hist_ylims = {
    'granule_cell': (0, 50),
    'MLI': (0, 150),
    'purkinje_cell': (0, 150),
    'dcn_cell_glut_large': (0, 100)
}

fig, axes = plt.subplots(num_cells, 3, figsize=(12, 2.8*num_cells), sharex=False, gridspec_kw={'width_ratios': [1.1, 1, 0.35]})

bin_size_ms = 20  # Histogram bin size set to 20ms
x_start_ms = 5000
x_end_ms = 6000
x_mid_ms = (x_start_ms + x_end_ms) // 2
x_ticks_ms = [x_start_ms, x_mid_ms, x_end_ms]
x_ticks_s = [t/1000 for t in x_ticks_ms]
x_ticklabels = [f'{t/1000:.1f}' for t in x_ticks_ms]

for i, cell in enumerate(main_cells):
    ax_raster = axes[i, 0]
    ax_hist = axes[i, 1]
    ax_bar = axes[i, 2]
    # For MLI, combine basket_cell and stellate_cell
    if cell == 'MLI':
        spikes_series_basket = spikes_single.get('basket_cell', None)
        spikes_series_stellate = spikes_single.get('stellate_cell', None)
        spikes_series_list = [s for s in [spikes_series_basket, spikes_series_stellate] if s is not None]
        right_indices = []
        for spikes_series in spikes_series_list:
            right_indices.extend([(spikes_series, idx) for idx in spikes_series.index if 'Right' in str(idx)])
        all_senders = []
        all_times = []
        for spikes_series, idx in right_indices:
            region_data = spikes_series[idx]
            senders = region_data.get('senders', [])
            times = region_data.get('times', [])
            all_senders.extend(senders)
            all_times.extend(times)
    else:
        spikes_series = spikes_single.get(cell, None)
        if spikes_series is None:
            continue
        right_indices = [idx for idx in spikes_series.index if 'Right' in str(idx)]
        all_senders = []
        all_times = []
        for idx in right_indices:
            region_data = spikes_series[idx]
            senders = region_data.get('senders', [])
            times = region_data.get('times', [])
            all_senders.extend(senders)
            all_times.extend(times)
    # Filter times to x_start_ms-x_end_ms ms
    filtered_senders = [s for s, t in zip(all_senders, all_times) if x_start_ms <= t < x_end_ms]
    filtered_times = [t for t in all_times if x_start_ms <= t < x_end_ms]
    unique_senders = np.unique(filtered_senders)
    # Only plot the first 50 cells
    unique_senders_raster = unique_senders[:50]
    sender_to_y = {sender: y+1 for y, sender in enumerate(unique_senders_raster)}
    for sender in unique_senders_raster:
        sender_times = [t for s, t in zip(filtered_senders, filtered_times) if s == sender]
        ax_raster.scatter(np.array(sender_times)/1000, np.full_like(sender_times, sender_to_y[sender]), color=cell_colors.get(cell, 'gray'), s=8, marker='|')
    ax_raster.set_yticks([])
    ax_raster.set_xlim(x_start_ms/1000, x_end_ms/1000)
    ax_raster.set_xticks(x_ticks_s)
    ax_raster.set_xticklabels(x_ticklabels)
    ax_raster.set_ylabel(" ")
    if i == num_cells - 1:
        ax_raster.set_xlabel('Time (s)')
    ax_raster.annotate(cell_names[cell], xy=(0, 1.08), xycoords='axes fraction', ha='left', va='bottom', fontsize=14)

    # Histogram plot (right)
    all_hist_times = []
    if cell == 'MLI':
        for spikes_series, idx in right_indices:
            region_data = spikes_series[idx]
            times = region_data.get('times', [])
            all_hist_times.extend(times)
    else:
        for idx in right_indices:
            region_data = spikes_series[idx]
            times = region_data.get('times', [])
            all_hist_times.extend(times)
    all_hist_times = np.array([t for t in all_hist_times if x_start_ms <= t < x_end_ms]).flatten()
    if all_hist_times.size > 0:
        min_time = x_start_ms
        max_time = x_end_ms
        bins = np.arange(min_time, max_time + bin_size_ms, bin_size_ms)
        bin_size_s = bin_size_ms / 1000.0
        n_neurons = len(unique_senders) if len(unique_senders) > 0 else 1
        counts, edges = np.histogram(all_hist_times, bins=bins)
        freq = counts / (bin_size_s * n_neurons)
        edges_s = edges[:-1] / 1000.0
        ax_hist.bar(edges_s, freq, width=bin_size_ms/1000.0, align='edge', color=cell_colors.get(cell, 'gray'), alpha=0.7)
    ax_hist.set_xlim(x_start_ms/1000, x_end_ms/1000)
    ax_hist.set_xticks(x_ticks_s)
    ax_hist.set_xticklabels(x_ticklabels)
    ax_hist.set_ylabel('Firing rate (Hz)')
    # Set specific y-limits for each cell type
    ax_hist.set_ylim(*hist_ylims[cell])
    if i == num_cells - 1:
        ax_hist.set_xlabel('Time (s)')

    # Point plot (right column): sim mean line + exp mean dot with min-max error bars
    color_cosim = viridis(0.75)  # Green, same as Co-sim line in PSD plot
    hw = 0.25  # half-width of horizontal line
    # Simulation: horizontal line at mean
    ax_bar.hlines(sim_values[i], -hw, hw, colors=color_cosim, linewidth=2, zorder=3,
                  label='Sim' if cell == 'granule_cell' else None)
    # Experimental: mean dot (smaller) with min-max error bars
    exp_yerr_lo = exp_means[i] - exp_mins[i]
    exp_yerr_hi = exp_maxs[i] - exp_means[i]
    ax_bar.errorbar(0, exp_means[i], yerr=[[exp_yerr_lo], [exp_yerr_hi]],
                    fmt='o', color=color_exp, capsize=5, markersize=5, linewidth=1.5,
                    label='Exp mean (min–max)' if cell == 'granule_cell' else None)
    if cell == 'granule_cell':
        ax_bar.legend(loc='upper left', fontsize=9, frameon=False)
    ax_bar.set_xlim(-0.5, 0.5)
    ax_bar.set_xticks([])
    ax_bar.set_ylabel('Firing rate (Hz)')
    if cell == 'granule_cell':
        ax_bar.set_ylim(0, 40)
    else:
        ax_bar.set_ylim(0, 200)
    if i == num_cells - 1:
        ax_bar.set_xlabel('')
    for spine in ['top', 'right']:
        ax_bar.spines[spine].set_visible(False)

raster_left = axes[0, 0].get_position().x0
fig.text(raster_left - 0.03, 0.5, '# neuron', va='center', rotation='vertical', fontsize=14)
hist_left = axes[0, 1].get_position().x0

plt.tight_layout(rect=[0.09, 0.04, 0.97, 0.97])

# Save figure to spikes_figures folder
fig_folder = os.path.join('Final10reps', 'coupling_and_spikes_figures')
os.makedirs(fig_folder, exist_ok=True)
fig_path = os.path.join(fig_folder, 'multi_panel_raster_hist_bar.png')
plt.savefig(fig_path, dpi=300)
save_figure_multi_format(fig, os.path.join(fig_folder, 'multi_panel_raster_hist_bar'))

plt.show()

In [ ]:
# Load and inspect iG06_nsd4_COSIM_tvb_serial_cosimulator.pkl
import os
from NESTlesions.file_utils import load_pickled_dict

res_file = os.path.join('Final10reps', 'COSIM_CEREBON_OFF', 'iG06_nsd4_COSIM_res_04_afferent_ts_trigeminal.pkl')
res_path = os.path.abspath(res_file)
print(f"Loading: {res_path}")

data = load_pickled_dict(res_file)
print(f"Keys in loaded data: {list(data.keys())}")


In [ ]:
# Explore the data structure
import numpy as np
import xarray as xr

print("Data structure exploration:")
print(f"Type of data: {type(data)}")

# Check if it's an xarray dataset or similar structure
if 'data' in data:
    actual_data = data['data']
    print(f"Type of actual data: {type(actual_data)}")
    print(f"Shape of data: {actual_data.shape if hasattr(actual_data, 'shape') else 'No shape attribute'}")

# Check dimensions
if 'dims' in data:
    print(f"Dimensions: {data['dims']}")

# Check coordinates
if 'coords' in data:
    print(f"Coordinates: {data['coords']}")

# Check attributes
if 'attrs' in data:
    print(f"Attributes: {data['attrs']}")

# If it's structured like an xarray, try to reconstruct it
if all(key in data for key in ['dims', 'coords', 'data']):
    try:
        # Reconstruct xarray DataArray
        da = xr.DataArray(
            data['data'],
            dims=data['dims'],
            coords=data['coords'],
            attrs=data.get('attrs', {}),
            name=data.get('name', 'data')
        )
        print(f"\nReconstructed DataArray:")
        print(f"Dimensions: {da.dims}")
        print(f"Coordinates: {list(da.coords.keys())}")
        print(f"Shape: {da.shape}")
        print(f"Data type: {da.dtype}")
        
        # Store the reconstructed data for plotting
        time_series_data = da
    except Exception as e:
        print(f"Error reconstructing xarray: {e}")
        time_series_data = data['data'] if 'data' in data else data
else:
    time_series_data = data

In [ ]:
# Plot summed cortical + subcortical coupling signal, ignoring thalamic
import matplotlib.pyplot as plt
import numpy as np
import os

# ===== USER SELECTION: choose which region to plot (0-indexed among non-thalamic regions) =====
selected_region = 3  # Change this to select a different region (see list printed below)
# ==============================================================================================

# Work directly with the numpy array to avoid coordinate issues
data_array = np.array(data['data'])

# Data shape: (Time, State Variable, Region, Neurons)
n_time, n_state_vars, n_regions, n_neurons = data_array.shape

print("Shape of data!", data_array.shape)

# Print the time vector (first dimension)
if 'coords' in data and 'Time' in data['coords']:
    time_coord = data['coords']['Time']
    if isinstance(time_coord, dict) and 'data' in time_coord:
        time_vector = np.array(time_coord['data'])
    else:
        time_vector = np.array(time_coord)
else:
    time_vector = np.arange(n_time)

# Check sampling rate and time step consistency
time_diffs = np.diff(time_vector)
mean_dt = np.mean(time_diffs)
min_dt = np.min(time_diffs)
max_dt = np.max(time_diffs)
print(f"\nTime step analysis:")
print(f"  Mean dt = {mean_dt:.6f}, Min dt = {min_dt:.6f}, Max dt = {max_dt:.6f}")
print(f"  All steps equal? {np.allclose(time_diffs, time_diffs[0])}")
print(f"  Step size = {time_diffs[0]:.6f} --> Sampling rate = {1.0 / time_diffs[0]:.2f} Hz")
if np.allclose(time_diffs, 1.0):
    print("  ✓ Confirmed: 1 ms steps (1000 Hz sampling rate)")
else:
    print(f"  ✗ Steps are NOT 1 ms! Actual step: {time_diffs[0]}")



# Create meaningful labels
regions = [f'Region {i+1}' for i in range(n_regions)]
state_vars = ['State Var 1', 'State Var 2', 'State Var 3'][:n_state_vars]

# Try to get better names from coordinates if available
if 'coords' in data:
    coords = data['coords']
    if 'Region' in coords:
        try:
            region_data = coords['Region']
            if isinstance(region_data, dict) and 'data' in region_data:
                region_values = region_data['data']
                if hasattr(region_values, '__iter__') and len(region_values) == n_regions:
                    regions = [str(r) for r in region_values]
        except Exception:
            pass
    if 'State Variable' in coords:
        try:
            sv_data = coords['State Variable']
            if isinstance(sv_data, dict) and 'data' in sv_data:
                sv_values = sv_data['data']
                if hasattr(sv_values, '__iter__') and len(sv_values) == n_state_vars:
                    state_vars = [str(s) for s in sv_values]
        except Exception:
            pass

# Filter out thalamic regions
non_thalamic_regions = []
non_thalamic_indices = []
for i, region in enumerate(regions):
    region_lower = str(region).lower()
    if 'thalamic' not in region_lower and 'thalamus' not in region_lower:
        non_thalamic_regions.append(region)
        non_thalamic_indices.append(i)

# Print available regions for user reference
print("Available non-thalamic regions:")
for idx, name in enumerate(non_thalamic_regions):
    marker = " <-- selected" if idx == selected_region else ""
    print(f"  {idx}: {name}{marker}")

# Validate selection
assert 0 <= selected_region < len(non_thalamic_indices), \
    f"selected_region={selected_region} is out of range [0, {len(non_thalamic_indices)-1}]"

region_idx = non_thalamic_indices[selected_region]
region_name = non_thalamic_regions[selected_region]
# Create a short region name for the filename (lowercase, spaces/special chars replaced by underscores)
region_name_short = region_name.strip().replace(' ', '_').replace('/', '_').replace('\\', '_')
region_name_short = ''.join(c if c.isalnum() or c == '_' else '' for c in region_name_short).lower()
print(f"\nPlotting region: {region_name}")

# Classify state variables into cortical and subcortical (exclude thalamic)
cortical_indices = []
cortical_labels = []
subcortical_indices = []
subcortical_labels = []

for i, sv in enumerate(state_vars):
    sv_lower = str(sv).lower()
    if 'subcortical' in sv_lower:
        subcortical_indices.append(i)
        subcortical_labels.append(sv)
    elif 'cortical' in sv_lower:
        cortical_indices.append(i)
        cortical_labels.append(sv)

# Time range: extend by 2/3 of the selected interval on each side
interval_ms = x_end_ms - x_start_ms
plot_start_ms = max(0, x_start_ms - 2 * interval_ms // 3)
plot_end_ms = x_end_ms + 2 * interval_ms // 3
start_idx = min(plot_start_ms, n_time - 1)
end_idx = min(plot_end_ms, n_time)
time_restricted = np.arange(start_idx, end_idx) / 1000.0

# Save figure folder
fig_folder = os.path.join('Final10reps', 'coupling_and_spikes_figures')
os.makedirs(fig_folder, exist_ok=True)

plt.close('all')

# ====== FIGURE: SUMMED (CORTICAL + SUBCORTICAL) COUPLING + dominant oscillation sinusoid ======
if cortical_indices and subcortical_indices:
    from scipy.optimize import curve_fit
    from scipy.signal import welch

    fig, ax = plt.subplots(figsize=(10, 3))

    c_idx = cortical_indices[0]
    s_idx = subcortical_indices[0]

    # Full signals (all time points) for frequency extraction
    if n_neurons == 1:
        full_cortical = data_array[:, c_idx, region_idx, 0]
        full_subcortical = data_array[:, s_idx, region_idx, 0]
    else:
        full_cortical = np.mean(data_array[:, c_idx, region_idx, :], axis=-1)
        full_subcortical = np.mean(data_array[:, s_idx, region_idx, :], axis=-1)

    # Sum cortical + subcortical
    full_signal_sum = full_cortical + full_subcortical

    # Plotted segment
    signal_sum = full_signal_sum[start_idx:end_idx]
    ax.plot(time_restricted, signal_sum, linewidth=2, alpha=0.8,
            color=viridis(0.55), label='cortical + subcortical')

    # --- Extract dominant oscillation frequency via PSD (Welch's method) on FULL summed signal ---
    fs = 1.0 / mean_dt  # Sampling rate derived from time vector
    nperseg = min(1000, len(full_signal_sum))
    psd_freqs, psd_power = welch(full_signal_sum, fs=fs, nperseg=nperseg, noverlap=nperseg // 2)

    # Ignore DC (0 Hz) and find peak frequency in PSD
    psd_power_no_dc = psd_power.copy()
    psd_power_no_dc[0] = 0
    peak_idx = np.argmax(psd_power_no_dc)
    dominant_freq = psd_freqs[peak_idx]
    dominant_power = psd_power[peak_idx]

    print(f"Dominant oscillation frequency (PSD): {dominant_freq:.2f} Hz  (power = {dominant_power:.6f})")

    # # --- Fit a sinusoid: A * sin(2*pi*f*t + phi) + offset ---
    # def sinusoid(t, A, phi, offset):
    #     return A * np.sin(2 * np.pi * dominant_freq * t + phi) + offset

    # A0 = np.sqrt(2 * dominant_power * (fs / nperseg))
    # phi0 = 0.0
    # offset0 = np.mean(signal_sum)
    # try:
    #     popt, _ = curve_fit(sinusoid, time_restricted, signal_sum, p0=[A0, phi0, offset0])
    #     fitted = sinusoid(time_restricted, *popt)
    #     ax.plot(time_restricted, fitted, 'r--', linewidth=1.5, alpha=0.9,
    #             label=f'Fit: {dominant_freq:.1f} Hz')
    #     print(f"Fitted sinusoid: A={popt[0]:.4f}, phase={np.degrees(popt[1]):.1f} deg, offset={popt[2]:.4f}")
    # except RuntimeError:
    #     fitted = A0 * np.sin(2 * np.pi * dominant_freq * time_restricted + phi0) + offset0
    #     ax.plot(time_restricted, fitted, 'r--', linewidth=1.5, alpha=0.9,
    #             label=f'PSD peak: {dominant_freq:.1f} Hz')
    #     print(f"Curve fit failed; using PSD estimate: A={A0:.4f}, freq={dominant_freq:.2f} Hz")

    ax.set_title(f'{region_name}', fontsize=14)
    ax.set_ylabel('Afferent coupling')
    ax.set_xlabel('Time (s)')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(plot_start_ms / 1000.0, plot_end_ms / 1000.0)
    ax.set_ylim(-10, 100)
    ax.set_xticks(np.arange(np.ceil(plot_start_ms / 500.0) * 0.5, plot_end_ms / 1000.0 + 0.01, 0.5))
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.legend(loc='upper right', fontsize=9)

    fig.tight_layout()
    fig_path = os.path.join(fig_folder, f'total_coupling_{region_name_short}.png')
    fig.savefig(fig_path, dpi=300)
    save_figure_multi_format(fig, os.path.join(fig_folder, f'total_coupling_{region_name_short}'))
    plt.show()
    print(f"Total coupling figure saved to {fig_path}")

# Summary statistics for the selected region
print(f"\n{'='*60}")
print("SUMMED (CORTICAL + SUBCORTICAL):")
if cortical_indices and subcortical_indices:
    print(f"  {region_name}: mean={np.mean(full_signal_sum):.4f}, std={np.std(full_signal_sum):.4f}")
for label, var_labels, idx_list in [('CORTICAL', cortical_labels, cortical_indices),
                                     ('SUBCORTICAL', subcortical_labels, subcortical_indices)]:
    if idx_list:
        print(f"{label}:")
        for j, var_idx in enumerate(idx_list):
            if n_neurons == 1:
                vals = data_array[:, var_idx, region_idx, 0]
            else:
                vals = np.mean(data_array[:, var_idx, region_idx, :], axis=-1)
            print(f"  {region_name} ({var_labels[j]}): mean={np.mean(vals):.4f}, std={np.std(vals):.4f}")
print(f"{'='*60}")